In [ ]:
# Run once, then restart the kernel. This covers all four notebooks.
%pip install -q pandas numpy geopandas pyogrio shapely osmnx networkx scipy scikit-learn openpyxl matplotlib gurobipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("..")      # the data files sit one folder above notebooks/
TRAFFIC = ROOT / "Automated_Traffic_Volume_Counts_20261104.csv"

BETA = 0.15                                    # how much extra traffic slows travel
REPORT_R = [1.000, 2.374, 2.942, 2.371]        # R_t in the submitted report
REPORT_KAPPA = [1.000, 1.206, 1.291, 1.206]    # kappa_t in the submitted report

# 01 Traffic congestion factor

We slow down ambulance travel times at busy hours with a factor kappa for each of four 6-hour periods. Here we work it out from NYC DOT's automated traffic counts, Manhattan only.

- Reads: `Automated_Traffic_Volume_Counts_20261104.csv`
- Writes: nothing. The kappa values used downstream (notebook 03) are the report's, which this notebook reproduces.

In [ ]:
cols = {"requestid", "boro", "yr", "m", "d", "hh", "mm", "vol", "segmentid", "direction"}
df = pd.read_csv(TRAFFIC, usecols=lambda c: c.strip().lower() in cols)    # skips the large map column
df.columns = [c.strip().lower() for c in df.columns]
print(df.shape)
print(df["boro"].value_counts())

mn = df[df["boro"].astype(str).str.strip().str.upper() == "MANHATTAN"].copy()
mn["year"] = np.where(mn["yr"] < 100, 2000 + mn["yr"], mn["yr"])          # years come as two digits
mn["date"] = pd.to_datetime(dict(year=mn["year"], month=mn["m"], day=mn["d"]), errors="coerce")
mn["slot"] = mn["hh"] * 60 + mn["mm"]                                      # minutes since midnight
mn["period"] = mn["hh"] // 6                                               # 0 = 00-06, 1 = 06-12, 2 = 12-18, 3 = 18-24
print("Manhattan rows:", len(mn), "| years:", mn["year"].min(), "to", mn["year"].max())

In [ ]:
# A count-day is one recorder, one street segment, one direction, one date.
key = ["requestid", "segmentid", "direction", "date"]
g = mn.groupby(key)["slot"].agg(n_rows="size", n_slots="nunique")
full_days = g.index[(g["n_rows"] == 96) & (g["n_slots"] == 96)]            # a full day has 96 fifteen-minute windows
print("count-days:", len(g), "| complete:", len(full_days), "| share:", round(len(full_days) / len(g), 3))

mn = mn.set_index(key).loc[full_days].reset_index()

Some recorders were only out for part of a day. We keep full days only. Partial days would make the later periods look quieter than they are and pull the ratios down.

In [ ]:
mn["vol"] = pd.to_numeric(mn["vol"].astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")
print("volumes that could not be read:", int(mn["vol"].isna().sum()))

daily = mn.pivot_table(index=key, columns="period", values="vol", aggfunc="sum")
daily.columns = ["P1", "P2", "P3", "P4"]
daily = daily[daily["P1"] > 0]                 # drop days with no overnight traffic (a broken counter)

R = daily.mean() / daily["P1"].mean()          # how busy each period is compared with overnight
kappa = 1 + BETA * (R - 1).clip(lower=0)

result = pd.DataFrame({"R (ours)": R.values, "R (report)": REPORT_R,
                       "kappa (ours)": kappa.values, "kappa (report)": REPORT_KAPPA},
                      index=["00-06", "06-12", "12-18", "18-24"])
print(result.round(3).to_string())
print("count-days used:", len(daily), "| street segments:", daily.reset_index()["segmentid"].nunique())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
x = np.arange(4)
for k, (col, ttl) in enumerate([("R", "Traffic relative to overnight (R)"), ("kappa", "Congestion factor (kappa)")]):
    ax[k].bar(x - 0.2, result[f"{col} (ours)"], 0.4, label="ours")
    ax[k].bar(x + 0.2, result[f"{col} (report)"], 0.4, label="report")
    ax[k].set_xticks(x, result.index)
    ax[k].set_title(ttl)
ax[0].legend()
plt.tight_layout()
plt.show()

Our R values are within about 5% of the report's, and kappa is off by 0.022 at most (12-18: 1.269 against 1.291). Notebook 03 uses the report's kappa so everything matches the submitted report.

R is a ratio of averages (average volume in the period over average overnight volume). Averaging each day's own ratio instead gives R of 6 to 9, because quiet streets with a handful of cars overnight blow the ratio up, so we don't use it.